# Downloading the librãry

In [2]:
pip install langgraph

  Using cached langgraph-1.0.5-py3-none-any.whl.metadata (7.4 kB)
  Using cached langchain_core-1.2.5-py3-none-any.whl.metadata (3.7 kB)
  Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langgraph_prebuilt-1.0.5-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached xxhash-3.6.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached ormsgpack-1.12.1-cp313-cp313-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
  Using cached orjson-3.11.5-cp313-cp313-macosx_15_0_arm64.whl.metadata (41 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.5.0-py3-none-any.whl.metadata (15 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached uuid_utils-0.12.0-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10

# Including the Library

In [37]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Making State 

In [38]:
class Batsman(TypedDict):
    runs : int
    fours : int
    sixs : int
    balls : int

    sr : float
    bpb : float
    boundary_percentage : float
    summary : str

# Making function for calculating Boundary percentage

### NOTE : Here we can't use return state because the state goes to parallel in three nodes and when the result goes to summany then langgraph thinks that there we will be some change in input attributes like runs,fours and then give error 
### So to teckle this we can return the particular dictionary instead of state also the function is taking the dictionary only so we can return it 

In [39]:
def calculate_boundary_percentage(state : Batsman):
    boundary_percentage = ((state['fours']*4+state['sixs']*6)/state['runs'])*100
    return {'boundary_percentage' : boundary_percentage}

# Making function for calculating bpb(ball per boundary)

In [40]:
def calculate_bpb(state : Batsman):
    bpb = state['balls']/(state['fours']+state['sixs'])
    return {'bpb' : bpb}

# Making function of calculating the str rate

In [48]:
def calculate_str(state : Batsman):
    sr = (state['runs']/state['balls'])*100
    return {'sr' : sr}

In [42]:
def calculate_summary(state : Batsman):
    summary = f"""
Strike rate :- {state['sr']}\n
Ball per boundary :- {state['bpb']} \n
Boundary Percentage :- {state['boundary_percentage']}\n

"""
    
    return {'summary' : summary}

# Making the Graph 

In [43]:
graph = StateGraph(Batsman)

# Nodes 

graph.add_node('calculate_str',calculate_str)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percentage',calculate_boundary_percentage)
graph.add_node('calculate_summary',calculate_summary)

# Edges

graph.add_edge(START,'calculate_str')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percentage')

graph.add_edge('calculate_str','calculate_summary')
graph.add_edge('calculate_bpb','calculate_summary')
graph.add_edge('calculate_boundary_percentage','calculate_summary')

graph.add_edge('calculate_summary',END)

# use this if you want to see the graph
# graph.compile()

workflow = graph.compile()

In [ ]:
workflow

In [49]:
initial_state={
    'runs' : 63,
    'fours' : 5,
    'sixs' : 5,
    'balls' : 25
    
    

}

workflow.invoke(initial_state)

{'runs': 63,
 'fours': 5,
 'sixs': 5,
 'balls': 25,
 'sr': 0.0252,
 'bpb': 2.5,
 'boundary_percentage': 79.36507936507937,
 'summary': '\nStrike rate :- 0.0252\n\nBall per boundary :- 2.5 \n\nBoundary Percentage :- 79.36507936507937\n\n\n'}